## Chapter: Standardized Soil Moisture Index (SSI) – Legacy MATLAB Methodology and v1 Framework

### 1. Conceptual Overview

The **Standardized Soil Moisture Index (SSI)** quantifies how unusual current soil moisture conditions are compared to typical conditions for the same time of year.  
Values near **0** indicate normal soil moisture; **negative** values indicate dryness; **positive** values indicate wetness.  
It is computed separately for each grid cell and each month, then standardized to a normal distribution.

---

### 2. The Reference and Target Time Series

To compute SSI, two time series are needed:

- **Reference series:** defines what is considered "normal" soil moisture.  
  It represents the long-term climatology or baseline distribution used for standardization.

- **Target series:** the period or scenario whose SSI we want to evaluate against that baseline.

In the ISIMIP attribution framework:

- Each of the **7 global hydrological models (GHMs)** is treated **independently**.  
- Within each model, SSI is computed separately for each of the **4 scenarios**:

  1. `obsclim_histsoc_default` — “factual” run (includes climate change and observed human forcing).  
  2. `counterclim_histsoc_default` — “counterfactual” run (no climate change, with historical human forcing).  
  3. `counterclim_1901soc_default` — (no climate change, pre-industrial human forcing).  
  4. `obsclim_1901soc_default` — (climate change only, no direct human forcing).

The **observational soil moisture datasets** are handled similarly, but their reference periods are limited by data availability.

---

### 3. Legacy MATLAB SSI Calculation (from `SSI_mod.m`)

The MATLAB routine implements a **non-parametric**, **month-wise**, **3-month aggregated** SSI following  
[Farahmand & AghaKouchak (2015)](https://doi.org/10.1016/j.advwatres.2014.11.012).  
Below is a simplified step-by-step explanation.

#### Step 1: Inputs
```matlab
SSI_mod(td_ref, td_proj, sc)
```


#### Step 2: Aggregation
Both the reference and target time series are converted to **3-month running sums** (trailing averages).  
This means that the SSI for a given month reflects soil moisture conditions over that month and the two preceding months.  
This smoothing removes short-term noise and emphasizes persistent wet or dry anomalies.

#### Step 3: Month-wise Standardization
To account for seasonality, the standardization is performed **separately for each calendar month (January to December)**:

1. Extract all values for a given month (e.g., all Januaries) from the reference (`d_ref`) and target (`d_proj`) time series.  
2. Compute **empirical ranks**:
   - For the reference data: rank each value within its own distribution.  
   - For the target data: rank each target value relative to the reference distribution (how many reference values are ≤ that value).  
3. Convert ranks to probabilities using the **plotting-position formula**:  
   $$
   p = \frac{\text{rank} - 0.44}{n_\text{ref} + 0.12}
   $$
   where $ n_\text{ref} $ is the number of reference samples for that month.  
4. Transform these probabilities into **standard normal scores** using the inverse normal CDF:  
   $$
   \text{SSI} = \Phi^{-1}(p)
   $$
   where $ \Phi^{-1} $ is the inverse of the cumulative normal distribution.  

This approach is **non-parametric**, meaning it does not assume any particular distribution (e.g., gamma or normal) for soil moisture.  
It standardizes the data purely based on empirical ranks.

#### Step 4: Output
The output `SI_proj` is the standardized monthly series (SSI) for the target period.  
The first `sc−1` months (here, 2 months) are undefined and set to NaN due to the 3-month aggregation window.

---

### 4. Drought Event Analysis (from `runtheo.m`)

Once the monthly SSI time series is available for a 30-year window, the MATLAB script identifies and characterizes individual **drought events**:

1. **Short wet interruptions (<6 months)** between dry spells are ignored — they are flipped slightly negative so that droughts remain continuous.  
2. **Weak negative runs (SSI > −1)** are excluded — only droughts that cross the threshold of −1 are kept.  
3. **Drought events** are then defined as consecutive months with SSI < 0.  
4. For each event, several descriptive metrics are computed:  
   - **D (Duration):** number of consecutive dry months.  
   - **M (Magnitude):** total dryness, i.e., the sum of |SSI| values across the event.  
   - **PI (Mean Intensity):** M divided by D — the average intensity per month.  
   - **DDD (Time to Drought Trough):** number of months from the start of the event to the minimum SSI value.  
   - **DRD (Recovery Duration):** months needed to recover after the trough (D − DDD).  
5. If no event meets the criteria, all metrics are set to NaN.

---

### 5. Attribution Framework

The ISIMIP experiment allows disentangling the contributions of different forcing types to soil-moisture changes.  
By comparing scenario pairs, the following effects can be quantified:

| Effect | Scenario Difference | Interpretation |
|:-------|:--------------------|:---------------|
| **Climate Change (CC)** | `obsclim_histsoc_default` − `counterclim_histsoc_default` | Isolates the effect of anthropogenic climate change while keeping human influences constant. |
| **Direct Human Forcing (DHF)** | `counterclim_histsoc_default` − `counterclim_1901soc_default` | Captures the effect of human land and water use, excluding climate change. |
| **Combined Effect** | `obsclim_histsoc_default` − `counterclim_1901soc_default` | Represents the joint impact of climate change and human activities. |

Each difference is computed for the drought metrics (D, M, PI, DDD, DRD) and mapped globally.

---

### 6. Limitations of the Legacy Implementation

While scientifically meaningful, the original MATLAB implementation introduced several methodological issues:

1. **Baseline contamination:** the target period was appended to the reference, meaning the evaluation window influenced its own baseline.  
2. **Over-pooled reference:** all scenarios and socioeconomic conditions were merged into a single reference distribution, blurring physical distinctions.  
3. **Inconsistent normalization:** the inclusion of evaluation data in the baseline leads to under-standardized SSI values.  
4. **Fixed month-wise ranking:** target probabilities use the reference sample size for normalization, slightly distorting the tails.  
5. **No explicit handling of zero-variance or missing months**, which may cause undefined SSI in some grids.

---

### 7. v1 Recommended Framework

The improved v1 methodology corrects these issues while preserving the non-parametric nature of the SSI:

1. Define a **clean, fixed reference period** (e.g., 1995–2014 for models; 1981–2010 for observations).  
2. Use **only that reference** to build monthly CDFs — never include the evaluation period.  
3. Apply the same **3-month trailing, non-parametric, month-wise normalization**.  
4. Compute SSI separately for each model and scenario; then compute ensemble median and spread.  
5. For observations, use their available years and compare overlapping periods with models.  
6. Use correlations and differences between model SSI and observed SSI for evaluation.

---

### 8. Summary Table

| Step | Legacy MATLAB | v1 Standardized Approach |
|------|----------------|--------------------------|
| Reference period | All scenarios concatenated | Fixed baseline (e.g., 1995–2014) |
| Target | 30-year slice per scenario | Same |
| SSI method | Non-parametric, 3-month trailing | Same |
| Drought threshold | SSI < −1 | Same |
| Issue | Target appended to reference | Removed |
| Output metrics | D, M, PI, DDD, DRD | Same, but unbiased |

---

**In summary:**  
The MATLAB framework correctly applies a non-parametric month-wise SSI and consistent drought metrics, but its baseline definition contaminates the standardization.  
The v1 pipeline maintains the original drought logic while introducing a fixed, uncontaminated reference period, ensuring reproducible and bias-free soil-moisture anomaly attribution.